In [1]:
import pandas as pd

In [3]:
%cd ad_gwas_hits

/Users/jonahpacis/Documents/CSE283/ad_classifier_exRNA/cse283_alzheimers_prediction_blood_exRNA/ad_gwas_hits


In [4]:
!pwd

/Users/jonahpacis/Documents/CSE283/ad_classifier_exRNA/cse283_alzheimers_prediction_blood_exRNA/ad_gwas_hits


In [5]:
df_verified = pd.read_csv("AD_GWAS_hits_gene_verified.csv")

In [6]:
df_risk = pd.read_csv("AD_GWAS_hits_risk.csv")

In [7]:
df_verified.head(10)

,Table 1: List of AD Loci with Genetic Evidence Compiled by ADSP Gene Verification Committee,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,Number,Chr,Location (hg38),SNV,Reported Gene/ Closest gene
1,1,1,109345810,rs141749679,SORT1
2,2,1,207577223,rs679515,CR1
3,3,2,9558882,rs72777026,ADAM17
4,4,2,37304796,rs17020490,PRKD3
5,5,2,105749599,rs143080277,NCK2
6,6,2,127135234,rs6733839,BIN1
7,7,2,202878716,rs139643391,WDR12
8,8,2,233117202,rs10933431,INPP5D
9,9,3,155069722,rs16824536,MME


In [14]:
def load_gwas_combined(verified_csv: str, risk_csv: str) -> pd.DataFrame:

    df_verified = pd.read_csv(verified_csv, header=1)
    df_risk     = pd.read_csv(risk_csv,     header=1)

    # ── Verified ──────────────────────────────────────────────────────────
    df_v = pd.DataFrame({
        "gene_symbol"   : df_verified["Reported Gene/ Closest gene"].astype(str).str.strip(),
        "chr"           : df_verified["Chr"].astype(str).str.strip(),
        "location_hg38" : pd.to_numeric(df_verified["Location (hg38)"], errors="coerce"),
        "snv"           : df_verified["SNV"].astype(str).str.strip(),
        "source"        : "verified",
    })

    # ── Risk ──────────────────────────────────────────────────────────────
    # Location is a range string e.g. "chr19:1,039,997-1,065,572"
    # Parse out chr and start position; no SNV column exists
    loc = df_risk["Location (hg38)"].astype(str).str.strip()
    chr_parsed = loc.str.extract(r"chr([^:]+):", expand=False)
    pos_parsed = (
        loc.str.extract(r":([0-9,]+)-", expand=False)
           .str.replace(",", "", regex=False)
    )
    df_r = pd.DataFrame({
        "gene_symbol"   : df_risk["Gene"].astype(str).str.strip(),
        "chr"           : chr_parsed,
        "location_hg38" : pd.to_numeric(pos_parsed, errors="coerce"),
        "snv"           : pd.NA,          # not available in risk table
        "source"        : "risk",
    })

    # ── Combine ───────────────────────────────────────────────────────────
    combined = pd.concat([df_v, df_r], ignore_index=True)

    # Drop placeholder/NaN gene symbols
    combined = combined[
        ~combined["gene_symbol"].isin(["nan", "NaN", ""])
    ].copy()

    # Split multi-gene cells e.g. "MS4A6A/MS4A4E" or "CLU,BIN1"
    combined = combined.assign(
        gene_symbol=combined["gene_symbol"].str.split(r"[/,]")
    ).explode("gene_symbol")
    combined["gene_symbol"] = combined["gene_symbol"].str.strip()
    combined = combined[combined["gene_symbol"].str.len() > 0]

    # Tag genes appearing in both lists (match on symbol since risk has no SNV)
    in_both = (
        set(df_v["gene_symbol"].str.strip()) &
        set(df_r["gene_symbol"].str.strip())
    )
    combined.loc[combined["gene_symbol"].isin(in_both), "source"] = "verified+risk"

    combined = combined.drop_duplicates(subset=["gene_symbol", "chr", "location_hg38"])
    combined = combined.reset_index(drop=True)

    print(f"Combined GWAS list: {len(combined)} entries, "
          f"{combined['gene_symbol'].nunique()} unique genes")
    print(combined["source"].value_counts().to_string())
    return combined


df_gwas = load_gwas_combined("AD_GWAS_hits_gene_verified.csv", "AD_GWAS_hits_risk.csv")
gwas_gene_set = set(df_gwas["gene_symbol"])

Combined GWAS list: 96 entries, 86 unique genes
source
verified         66
verified+risk    20
risk             10


In [16]:
df_gwas

,gene_symbol,chr,location_hg38,snv,source
0,SORT1,1,109345810,rs141749679,verified
1,CR1,1,207577223,rs679515,verified+risk
2,ADAM17,2,9558882,rs72777026,verified
3,PRKD3,2,37304796,rs17020490,verified
4,NCK2,2,105749599,rs143080277,verified
...,...,...,...,...,...
91,RIN3,14,92513781,<NA>,risk
92,SORL1,11,121452314,<NA>,verified+risk
93,SPI1,11,47354860,<NA>,verified+risk
94,TREM2,6,41158506,<NA>,verified+risk


In [17]:
df_gwas.to_csv("AD_GWAS_hits.csv")